# 01. Sales Baseline, Schema Audit & Time-Series Decomposition

**Objective:** Establish clean daily, hourly (`06:00–21:00`), and 4-way SKU baseline profiles prior to external incident correlation or ROI simulation.

**Inputs / Assumptions:**
- `BGA_DATA_MODE="sample"` (default): synthetic CSVs in `data/sample/`.
- `BGA_DATA_MODE="raw"`: read-only allowlisted SQLite loaders (`src.db_loader`) against `data/raw/master_sales.db`.

**How to Read This Notebook:** Inspect coverage audit -> day-of-week & *quincena* decomposition -> weekday vs. weekend hourly demand -> SKU category mix.

## 1. Environment Setup & Rendering Helpers

**Objective:** Resolve `REPO_ROOT`, isolate `MPLCONFIGDIR` per process PID, configure interactive/inline plotting safely, and load `StoreConfig`.

**Inputs / Assumptions:** Reads `BGA_DATA_MODE`, optional `data/raw/local_assumptions.json`, and `BGA_*` environment overrides.

**How to Read the Output:** Confirms active data mode, branch ID, and coordinate verification status.

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path(os.environ.get("BGA_REPO_ROOT", Path.cwd())).resolve()
if not (REPO_ROOT / "src").is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd

from src.config import load_config
from src.notebook_utils import (
    compute_weighted_dow_summary,
    configure_mpl_env,
    decompose_daily_baseline,
    render_table,
    show_fig,
    split_weekday_weekend_hourly,
)

plt = configure_mpl_env()
cfg = load_config(repo_root=REPO_ROOT)
DATA_MODE = cfg.data_mode
print(f"DATA_MODE={DATA_MODE} | branch={cfg.branch_id} | coords=({cfg.lat}, {cfg.lon}) verified={cfg.coords_verified}")

## 2. Schema & Dataset Coverage Audit

**Objective:** Load daily, hourly, and product-mix tables with explicit file and non-empty assertions in `raw` mode.

**Inputs / Assumptions:** `master_sales.db` must exist on disk when `DATA_MODE == "raw"`; hourly data is clamped to `cfg.operating_hours` (`06:00–21:00`).

**How to Read the Output:** Verify row counts, date ranges, and active hours match expected coverage before reading downstream charts.

In [ ]:
if DATA_MODE == "raw":
    from src.db_loader import inspect_schema, load_daily_sales, load_hourly_sales, load_product_mix
    if not cfg.master_sales_db_path.is_file():
        raise FileNotFoundError(f"Missing raw SQLite DB: {cfg.master_sales_db_path}")
    schema = inspect_schema(cfg.master_sales_db_path, db_kind="master_sales")
    daily_df = load_daily_sales(config=cfg)
    hourly_df = load_hourly_sales(config=cfg, complete_grid=True)
    mix_df = load_product_mix(config=cfg)
    if daily_df.empty or hourly_df.empty or mix_df.empty:
        raise ValueError("One or more allowlisted POS tables returned zero rows in raw mode.")
    source_label = f"SQLite allowlist ({len(schema)} tables)"
else:
    daily_df = pd.read_csv(cfg.sample_dir / "daily_sales.csv", parse_dates=["date"])
    hourly_df = pd.read_csv(cfg.sample_dir / "hourly_sales.csv", parse_dates=["date"])
    mix_df = pd.read_csv(cfg.sample_dir / "product_mix.csv", parse_dates=["date"])
    source_label = "Synthetic CSVs (data/sample)"

coverage_tbl = pd.DataFrame([
    {"dataset": "daily_sales", "rows": len(daily_df), "date_min": str(daily_df["date"].min().date()), "date_max": str(daily_df["date"].max().date()), "source": source_label},
    {"dataset": "hourly_sales", "rows": len(hourly_df), "date_min": str(hourly_df["date"].min().date()), "date_max": str(hourly_df["date"].max().date()), "source": source_label},
    {"dataset": "product_mix", "rows": len(mix_df), "date_min": str(mix_df["date"].min().date()), "date_max": str(mix_df["date"].max().date()), "source": source_label},
])
render_table(coverage_tbl, title="Dataset Coverage & Schema Audit", fmt={"rows": "{:,}"})

## 3. Day-of-Week Weighted Economics & Additive Baseline Decomposition

**Objective:** Compute volume-weighted average ticket (`total_revenue / total_tickets`) by day of week and decompose daily revenue into 14-day centered trend, DOW seasonality, *quincena* payday effect, and percentage residuals.

**Inputs / Assumptions:** *Quincena* window covers calendar days `{1, 14, 15, 16, 28, 29, 30, 31}`; vertical dashed line marks `cfg.tablet_channel_start_date` (`pos_channel_change`).

**How to Read the Output:** Compare weekday vs. weekend ticket weight, check pay-cycle lift, and inspect negative residual spikes for candidate external-shock dates.

In [ ]:
dow_summary = compute_weighted_dow_summary(daily_df)
decomp_df, pay_tbl = decompose_daily_baseline(daily_df)

render_table(
    dow_summary[["dow", "days", "mean_revenue", "mean_tickets", "weighted_avg_ticket"]],
    title="Day-of-Week Summary (Volume-Weighted Avg Ticket)",
    fmt={"mean_revenue": "{:,.2f}", "mean_tickets": "{:.1f}", "weighted_avg_ticket": "{:,.2f}"},
)
render_table(
    pay_tbl[["pay_cycle", "days", "mean_revenue", "mean_tickets", "weighted_avg_ticket", "pct_diff_vs_non_quincena"]],
    title="Pay-Cycle (Quincena vs Non-Quincena) Comparison",
    fmt={"mean_revenue": "{:,.2f}", "mean_tickets": "{:.1f}", "weighted_avg_ticket": "{:,.2f}", "pct_diff_vs_non_quincena": "{:.1%}"},
)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(decomp_df["date"], decomp_df["revenue"], alpha=0.45, label="Daily Revenue")
axes[0].plot(decomp_df["date"], decomp_df["trend"], linewidth=2.2, color="#d95f02", label="14-Day Centered Trend")
axes[0].axvline(pd.Timestamp(cfg.tablet_channel_start_date), linestyle="--", color="gray", label="Tablet Channel Audit Date")
axes[0].set_title(f"Daily Revenue Baseline & Trend ({cfg.currency})")
axes[0].set_ylabel(cfg.currency)
axes[0].legend()

axes[1].plot(decomp_df["date"], decomp_df["residual_pct"] * 100.0, color="#1b9e77", linewidth=1.2, label="Residual (%)")
axes[1].axhline(0.0, color="black", linestyle="--", linewidth=0.8)
axes[1].set_title("Additive Decomposition Residuals (Post Trend + DOW + Quincena)")
axes[1].set_ylabel("Residual (%)")
axes[1].legend()
show_fig(fig)

## 4. Weekday vs. Weekend Hourly Demand Profile

**Objective:** Separate weekday commuter curves from weekend leisure traffic across operating hours (`06:00–21:00`) and evaluate `P90`/`P95` congestion ceilings.

**Inputs / Assumptions:** Peak windows `cfg.peak_hours` are shaded/highlighted.

**How to Read the Output:** Large gaps between `mean_volume` and `p95_volume` during shaded peak hours signal latent queue capacity.

In [ ]:
ww_hourly = split_weekday_weekend_hourly(hourly_df, peak_hours=cfg.peak_hours)
peak_tbl = ww_hourly[ww_hourly["is_peak"]].copy()
fmt_hourly = {"mean_volume": "{:,.2f}", "p90_volume": "{:,.2f}", "p95_volume": "{:,.2f}", "mean_tickets": "{:.1f}"}
render_table(peak_tbl, title="Peak-Hour Demand Summary (Weekday vs Weekend)", fmt=fmt_hourly)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, dtype in zip(axes, ["Weekday", "Weekend"]):
    sub = ww_hourly[ww_hourly["day_type"] == dtype]
    colors = ["#d95f02" if pk else "#1b9e77" for pk in sub["is_peak"]]
    ax.bar(sub["hour_int"], sub["mean_volume"], color=colors, alpha=0.85, label="Mean Volume")
    ax.plot(sub["hour_int"], sub["p95_volume"], color="#2b8cbe", marker="o", linewidth=1.8, label="P95 Ceiling")
    ax.set_title(f"{dtype} Hourly Demand (Peak Hours Orange)")
    ax.set_xlabel("Hour of Day")
    ax.legend()
axes[0].set_ylabel(f"Hourly Revenue ({cfg.currency})")
show_fig(fig)

## 5. 4-Way Product Mix Summary

**Objective:** Measure unit share (and revenue share when available) across `Beverages`, `Food_Pastries`, `Combos`, and `Retail`.

**Inputs / Assumptions:** In `raw` mode, `Items` coverage extends through `cfg.items_coverage_end_date` (`2026-08-20`).

**How to Read the Output:** Establishes food and combo attach baselines used downstream in Notebook `05`.

In [ ]:
agg_mix = {"quantity_sold": ("quantity_sold", "sum")}
if "revenue" in mix_df.columns:
    agg_mix["revenue"] = ("revenue", "sum")
mix_summary = mix_df.groupby("category", as_index=False).agg(**agg_mix).sort_values("quantity_sold", ascending=True)
mix_summary["unit_share_pct"] = mix_summary["quantity_sold"] / mix_summary["quantity_sold"].sum()
if "revenue" in mix_summary.columns and mix_summary["revenue"].sum() > 0:
    mix_summary["rev_share_pct"] = mix_summary["revenue"] / mix_summary["revenue"].sum()

render_table(
    mix_summary.sort_values("quantity_sold", ascending=False),
    title="Product Mix by Category",
    fmt={"quantity_sold": "{:,.0f}", "revenue": "{:,.2f}", "unit_share_pct": "{:.1%}", "rev_share_pct": "{:.1%}"},
)

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.barh(mix_summary["category"], mix_summary["unit_share_pct"] * 100.0, color="#3288bd")
ax.set_xlabel("Share of Units Sold (%)")
ax.set_title("4-Way Product Mix Share")
show_fig(fig)